# Script for adding the `{{NWBib.de}}` template

NWBib is the bibliography of North Rhine-Westphalia, see: https://nwbib.de/. It has its own spatial taxonomy in order to assign a publication to a specific resource.
You can use the taxonomy as filter for the search in NWBib as well as for browsing: https://nwbib.de/spatial

This script should add the Vorlage:NWBib.de to pages in the German Wikipedia that are about a place that matches a place from the NWBib spatial taxonomy.

This should be achieved by querying wikidata for items that have a statement with property P6814 as well as a German Wikipedia page.

## Import pywikibot and rdflib

In [ ]:
import sys
!{sys.executable} -m pip install rdflib

In [ ]:
import pywikibot
import rdflib
from pywikibot import textlib
import time
import re

## Query Wikidata for Wikipedia pages of NWBib spatial concepts/locations

The [NWBib spatial classification](https://nwbib.de/spatial) has a corresponding [wikidata property](https://www.wikidata.org/wiki/Property:P6814) to mark entries that are covered in the classification. The following [SPARQL query searches for items that have both an NWBib spatial id as well as a German Wikipedia page](https://query.wikidata.org/index.html#%23%20Liste%20alle%20Wikidata-Items%20mit%20einer%20NWBib-ID%20(P6814)%0A%0APREFIX%20wd%3A%20%3Chttp%3A%2F%2Fwww.wikidata.org%2Fentity%2F%3E%0APREFIX%20wdt%3A%20%3Chttp%3A%2F%2Fwww.wikidata.org%2Fprop%2Fdirect%2F%3E%0APREFIX%20schema%3A%20%3Chttp%3A%2F%2Fschema.org%2F%3E%0ASELECT%20%3FarticleName%0AWHERE%20%7B%0A%20%20%20%20SERVICE%20%3Chttps%3A%2F%2Fquery.wikidata.org%2Fsparql%3E%20%7B%0A%20%20%20%20%20%20%3Fitem%20wdt%3AP6814%20%3FnwbibId.%0A%20%20%20%20%20%20%3Farticle%20schema%3Aabout%20%3Fitem.%0A%20%20%20%20%20%20%3Farticle%20schema%3Aname%20%3FarticleName%20.%0A%20%20%20%20%20%20%3Farticle%20schema%3AisPartOf%20%3Chttps%3A%2F%2Fde.wikipedia.org%2F%3E.%0A%20%20%20%20%7D%0A%20%20%7D).


In [ ]:
g = rdflib.Graph()
qres = g.query(
        """
PREFIX wd: <http://www.wikidata.org/entity/>
PREFIX wdt: <http://www.wikidata.org/prop/direct/>
PREFIX schema: <http://schema.org/>
SELECT ?articleName
WHERE {
    SERVICE <https://query.wikidata.org/sparql> {
      ?item wdt:P6814 ?nwbibId.
      ?article schema:about ?item.
      ?article schema:name ?articleName .
      ?article schema:isPartOf <https://de.wikipedia.org/>.
    }
  }
    """
)


## Iterate over the query results and add Vorlage:NWBib.de where necessary

The limit of intended max number of changes can be set with the variable `n`. When introducing changes to the bot script set the variable to a low number in order to test and review the changes. Only if your reviews catch no problems increase the number over 100 or even 1000.

In [ ]:
# Set number of intenden changes

n = 750

timestr = time.strftime("%Y%m%d-%H%M%S")
filePathAndPrefix = "logs/" + timestr + "-"

# Open all txt files, and let them append the writing for logs
file1 = open(filePathAndPrefix + "new.txt", "a")
file2 = open(filePathAndPrefix + "missingWikipediaPage.txt", "a")
file3 = open(filePathAndPrefix + "alreadyLinkedPage.txt", "a")
file4 = open(filePathAndPrefix + "protected.txt", "a")
file5 = open(filePathAndPrefix + "errors.txt", "a")
file6 = open(filePathAndPrefix + "report.txt", "a")

site = pywikibot.Site("de","wikipedia")
counter = 0
counterChangedTemplate = 0
counterAddedLinks = 0
counterMissingWikipediaPage = 0
counterAlreadyLinked = 0
counterPagesMissingWeblinkSection = 0
counterPagesMissingEinzelnachweiseSection = 0
counterProtected = 0
countErrors = 0

for row in qres:
    page = pywikibot.Page(site, row.articleName)
    pageExists = page.exists()
    counter += 1
    if counterAddedLinks < n:
        if  pageExists == True:
            if page.isRedirectPage():
                # Print and write redirect pages
                file2.write("Seite für " + row.articleName + " is only a redirect to an Wikipedia Page \n")
                counterMissingWikipediaPage += 1
            else:
                pageText = page.get()
                protected = page.protection()
                if protected == {}: 
                    if "{{NWBib.de}}" not in pageText:
                        withWeblinks = textlib.does_text_contain_section(page.text, "Weblinks")
                        withEinzelnachweise = textlib.does_text_contain_section(page.text, "Einzelnachweise")
                        # Replace old Vorlage if exists
                        if "{{NWBib}}" in pageText:
                            new_content = page.text.replace(str("{{NWBib}}"), str("{{NWBib.de}}")) # replace the old Vorlage with the correct one
                            page.put(new_content, summary='Bot: Ersetze Vorlage:NWBib durch Vorlage:NWBib.de') # Save
                            file1.write(row.articleName + " Replaced Vorlage:NWBib with Vorlage:NWBib.de\n")
                            print(page.title() + " Replaced Vorlage:NWBib with Vorlage:NWBib.de\n")
                            counterChangedTemplate += 1
                        # Add Vorlage:NWBib.de to existing Weblinks, as first link after the sister project links in order to no mix them in the 
                        elif withWeblinks == True:
                            try:
                                if re.search(r"(\=\= Weblinks \=\=(?:\n\{\{.+\}\}){0,}\n)",page.text):
                                    new_content = re.sub(r"(\=\= Weblinks \=\=(?:\n\{\{.+\}\}){0,}\n)",r"\g<1>* {{NWBib.de}}\n", page.text)
                                    page.put(new_content, summary='Bot: Ergänze Link zur NWBib mit Vorlage:NWBib.de') # Save
                                    file1.write(page.title() + " Vorlage:NWBib.de added \n")
                                    print(page.title() + " Vorlage:NWBib.de added \n")
                                    counterAddedLinks += 1
                                else:
                                    file5.write(page.title() + " Weblinks Regex not matching \n")
                                    print(page.title() + " Weblinks Regex not matching \n")
                            except ValueError:
                                file5.write(page.title() + " VlaueError with Weblinks \n")
                                print(page.title() + " ValueError with Weblinks \n")
        
                        # Add Vorlage:NWBib.de and section Weblinks if Einzelnachweise exist
                        elif withEinzelnachweise == True:
                            try:
                                if re.search(r"(\=\= Einzelnachweise \=\=)",page.text):
                                    webLinkContentAndEinzelnachweise ="== Weblinks ==\n* {{NWBib.de}}\n== Einzelnachweise ==" # prepend Weblink Section with the Vorlage:NWBib.de
                                    new_content = page.text.replace(str("== Einzelnachweise =="), str(webLinkContentAndEinzelnachweise)) # replace the existing Einzelnachweise content with the new Weblink+Einzelnachweise content
                                    page.put(new_content, summary='Bot: Ergänze Weblinks-Abschnitt und Link zur NWBib mit Vorlage:NWBib.de') # Save
                                    file1.write(page.title() + " Vorlage:NWBib.de added (no WebLinks) \n")
                                    print(page.title() + " Vorlage:NWBib.de added (no WebLinks) \n")
                                    counterPagesMissingWeblinkSection += 1
                                    counterAddedLinks += 1
                                else:
                                    file5.write(page.title() + " Einzelnachweise Regex not matching \n")
                                    print(page.title() + " Einzelnachweise Regex not matching \n")
                                    countErrors += 1
                            except ValueError:
                                file5.write(page.title() + " VlaueError with Einzelnachweis \n")
                                print(page.title() + " ValueError with Einzelnachweis \n")
                                countErrors += 1
                        # Add Vorlage:NWBib.de and section Weblinks if Einzelnachweise not exist
                        # Version 1 with Navigationsleiste
                        elif re.search(r"(\{\{Navigationsleiste.+(?:\n.*){0,}\[\[Kategorie.+]])",page.text):
                            new_content = re.sub(r"(\{\{Navigationsleiste.+(?:\n.*){0,}\[\[Kategorie.+]])",r"== Weblinks ==\n* {{NWBib.de}}\n\n\g<1>", page.text)
                            page.put(new_content, summary='Bot: Ergänze Weblinks-Abschnitt und Link zur NWBib mit Vorlage:NWBib.de') # Save
                            file1.write(page.title() + " Vorlage:NWBib.de added (no WebLinks + no Einzelnachweise) \n")
                            print(page.title() + " Vorlage:NWBib.de added (no WebLinks + no Einzelnachweise)  \n")
                            counterPagesMissingWeblinkSection += 1
                            counterPagesMissingEinzelnachweiseSection += 1
                            counterAddedLinks += 1 
                        # Version 2 just appending
                        else:
                            try:
                                new_content = textlib.add_text(page.text,"\n== Weblinks ==\n* {{NWBib.de}}\n")
                                page.put(new_content, summary='Bot: Ergänze Weblinks-Abschnitt und Link zur NWBib mit Vorlage:NWBib.de') # Save
                                file1.write(page.title() + " Vorlage:NWBib.de added (no WebLinks + no Einzelnachweise) \n")
                                print(page.title() + " Vorlage:NWBib.de added (no WebLinks + no Einzelnachweise)  \n")
                                counterPagesMissingWeblinkSection += 1
                                counterPagesMissingEinzelnachweiseSection += 1
                                counterAddedLinks += 1
                            except:
                                file5.write(row.articleName + " Error check it out \n")
                                print(row.articleName + " Error check it out  \n")
                                countErrors += 1
                    else:
                        # Print and write existing links
                        file3.write(row.articleName + " Vorlage:NWBib.de already exists \n")
                        counterAlreadyLinked += 1
                else:
                    if "{{NWBib.de}}" not in pageText:
                        # Print and write protected pages
                        file4.write(row.articleName + " is protected \n")
                        print(page.title() + " is protected \n")
                        counterProtected += 1
                    else:
                        # Print and write existing links
                        file3.write(row.articleName + " Vorlage:NWBib.de already exists in protected page \n")
                        counterAlreadyLinked += 1
        else:
                # Print and write missing pages
                file2.write("Seite für " + row.articleName + " is missing Wikipedia Page \n")
                counterMissingWikipediaPage += 1
    else:
        break

# Report of the results
file6.write("Processed Wikidata entries:" + str(counter) + "\n")
print("Processed Wikidata entries:" + str(counter))
file6.write("Newly added nwbib links:" + str(counterAddedLinks) + "\n")
print("Newly added nwbib links:" + str(counterAddedLinks))
file6.write("Changed Vorlage:" + str(counterChangedTemplate) + "\n")
print("Changed Vorlage:" + str(counterChangedTemplate))
file6.write("Missing weblink section:" + str(counterPagesMissingWeblinkSection) + "\n")
print("Missing weblink section:" + str(counterPagesMissingWeblinkSection))
file6.write("Missing Einzelnachweise section:" + str(counterPagesMissingEinzelnachweiseSection) + "\n")
print("Missing Einzelnachweise section:" + str(counterPagesMissingEinzelnachweiseSection))
file6.write("Already linked pages:" + str(counterAlreadyLinked) + "\n")
print("Already linked pages:" + str(counterAlreadyLinked))
file6.write("Missing Wikipedia Pages for Wikidata entry:" + str(counterMissingWikipediaPage) + "\n")
print("Missing Wikipedia Pages for Wikidata entry:" + str(counterMissingWikipediaPage))
file6.write("Errors:" + str(countErrors) + "\n")
print("Errors:" + str(countErrors))
file6.write("Protected pages:" + str(counterProtected) + "\n")
print("Protected pages:" + str(counterProtected))

file1.close()
file2.close()
file3.close()
file4.close()
file5.close()
file6.close()